# 03 - Transformer Architecture
Builds the full block and verifies residual connections.

In [1]:
import torch
from model.config import GPTConfig
from model.blocks import TransformerBlock

CFG = GPTConfig(n_layer=2, n_head=4, n_embd=64, block_size=16, vocab_size=100)
block = TransformerBlock(CFG)

## Residual connections - the gradient highway

In [2]:
torch.manual_seed(0)
x = torch.randn(1, 8, 64)
out = block(x)
print(f"Input  std:  {x.std().item():.3f}")
print(f"Output std:  {out.std().item():.3f}")
# If residuals work, output scale should be similar to input scale

Input  std:  1.016
Output std:  1.046


## Parameter count breakdown

In [3]:
from model.gpt import GPT
from model.config import TINY_CONFIG, GPT2_CONFIG

for name, cfg in [("Tiny (~20M)", TINY_CONFIG), ("GPT-2 small (124M)", GPT2_CONFIG)]:
    model = GPT(cfg)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n{name}: {total/1e6:.1f}M total params")
    for n, m in model.named_children():
        p = sum(x.numel() for x in m.parameters())
        print(f"  {n:<12} {p/1e6:.2f}M")


Tiny (~20M): 30.0M total params
  wte          19.30M
  wpe          0.10M
  drop         0.00M
  blocks       10.65M
  ln_f         0.00M
  lm_head      19.30M



GPT-2 small (124M): 124.0M total params
  wte          38.60M
  wpe          0.39M
  drop         0.00M
  blocks       85.05M
  ln_f         0.00M
  lm_head      38.60M
